In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sys
import os
from sklearn.preprocessing import LabelEncoder


project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.nbaPlayerLogs import NBAGameLogs
from src.features.feature_engineer.pbp_features import *

pd.set_option('display.max_columns', None)

## Fetches Player Gamelogs

In [ ]:
# df = NBAGameLogs(season='2024-25', season_type='Regular Season').fetch(skip_start_positions=True).build().get_df()
# df.head()

# Session by session — run this repeatedly until all games are done
logs = NBAGameLogs(season='2025-26', season_type='Playoffs')
logs.fetch(batch_size=50, start_position_delay=2.5, start_position_workers=5, checkpoint_path='tracking_checkpoint.csv')

In [15]:
s22 = pd.read_csv(project_root / 'data/raw/season_stats/S22.csv').sort_values(by='GAME_DATE')
p22 = pd.read_csv(project_root / 'data/raw/playoff_stats/P22.csv').sort_values(by='GAME_DATE')
s22 = pd.concat([s22, p22])
s22.head(5)

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED
0,0,0,0,26038,2021-22,1627761,DeAndre' Bembry,DeAndre',1610612751,BKN,Brooklyn Nets,22100001,2021-10-19,BKN @ MIL,L,3.750000,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,-9,0.0,0,0,0.0,1,3:45,1,38.7,37.5,37.5,133.3,150.0,150.0,-94.7,-112.5,-112.5,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.000,0.000,107.26,102.40,85.33,102.40,0.000,8,0.0,0.0,NaN,3.87,0.27,0,0,0,3,0,0,3,0,0,0.0,0,0,0.0,1,1,1.000,37,84,0.440,17,32,0.531,13,23,0.565,5,39,44,19,13.0,3,9,9,17,19,104,-23.0,101.8,102.0,117.7,124.5,-15.8,-22.5,0.514,1.46,14.8,0.176,0.707,0.459,0.127,0.542,0.552,105.0,102.0,85.0,102,0.407,1610612749,MIL,Milwaukee Bucks,48,105,0.457,17,45,0.378,14,18,0.778,13,41,54,25,8.0,8,9,9,19,17,127,23.0,117.7,124.5,101.8,102.0,15.8,22.5,0.521,3.13,17.4,0.293,0.824,0.541,0.078,0.538,0.562,105.0,102.0,85.0,102,0.593,0,SF,27.0,1.5,233.5,-1.5,234.0,0,0.000000,0.00000,0.000000,0,3
26,1,26,1,26036,2021-22,1630560,Cam Thomas,Cam,1610612751,BKN,Brooklyn Nets,22100001,2021-10-19,BKN @ MIL,L,3.750000,0,2,0.0,0,1,0.0,2,2,1.0,0,0,0,0,1,0,0,0,0,1,2,-9,1.0,0,0,2.0,1,3:45,1,38.7,37.5,37.5,133.3,150.0,150.0,-94.7,-112.5,-112.5,0.000,0.0,0.0,0.000,0.000,0.000,25.0,25.8,0.0,0.347,0.500,0.500,107.26,102.40,85.33,102.40,-0.105,8,0.0,2.0,NaN,4.02,0.28,0,1,1,6,0,0,2,0,0,0.0,0,2,0.0,1,3,0.333,37,84,0.440,17,32,0.531,13,23,0.565,5,39,44,19,13.0,3,9,9,17,19,104,-23.0,101.8,102.0,117.7,124.5,-15.8,-22.5,0.514,1.46,14.8,0.176,0.707,0.459,0.127,0.542,0.552,105.0,102.0,85.0,102,0.407,1610612749,MIL,Milwaukee Bucks,48,105,0.457,17,45,0.378,14,18,0.778,13,41,54,25,8.0,8,9,9,19,17,127,23.0,117.7,124.5,101.8,102.0,15.8,22.5,0.521,3.13,17.4,0.293,0.824,0.541,0.078,0.538,0.562,105.0,102.0,85.0,102,0.593,0,SG,20.0,1.5,233.5,-1.5,234.0,0,0.533333,0.00000,0.000000,0,4
27,2,27,2,26035,2021-22,1628975,Jevon Carter,Jevon,1610612751,BKN,Brooklyn Nets,22100001,2021-10-19,BKN @ MIL,L,18.950000,0,3,0.0,0,2,0.0,0,0,0.0,0,2,2,0,1,0,0,0,1,0,0,-29,1.4,0,0,2.0,1,18:57,1,72.0,72.5,72.5,133.6,148.7,148.7,-61.6,-76.2,-76.2,0.000,0.0,0.0,0.000,0.100,0.044,25.0,25.0,0.0,0.000,0.093,0.097,105.98,100.05,83.38,100.05,-0.043,40,0.0,3.0,NaN,4.

In [16]:
import sys
sys.path.append('src')
from src.utils.pbp_stats import NBAPlayByPlay

# Pass a list of game IDs
game_ids = s22['GAME_ID'].unique().tolist()

pbp = NBAPlayByPlay(game_ids).fetch(
    delay=2,           # seconds between requests (default 0.3)
    workers=5,           # parallel threads (default 8)
    batch_size=100,      # games per batch (default 100)
    checkpoint_path='pbp_checkpoint.csv',  # saves progress so you can resume
    run_all_batches=True # set False to stop after first batch
).build()
df = pbp.get_df()
df.to_csv(project_root / 'data/raw/pbp_season_stats/S22.csv', index=False)
df


✓ Checkpoint loaded — 1317/1317 games done, 0 remaining
✓ All games already fetched from checkpoint
✓ Built — shape: (642477, 24)


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022100006,2,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.0,0.0,0,NaN,Start of 1st Period (7:47 PM EST),period,start,0,0,1
1,0022100006,4,PT12M00.00S,1,1610612761,TOR,1630173,Achiuwa,P. Achiuwa,0,0,0,NaN,0,NaN,NaN,0,h,Jump Ball Achiuwa vs. Gafford: Tip to Dragic,Jump Ball,NaN,1,0,2
2,0022100006,7,PT11M46.00S,1,1610612761,TOR,1627832,VanVleet,F. VanVleet,-164,221,28,Made,1,3.0,0.0,3,h,VanVleet 28' 3PT Jump Shot (3 PTS) (Dragic 1 AST),Made Shot,Jump Shot,1,3,3
3,0022100006,9,PT11M24.00S,1,1610612764,WAS,203484,Caldwell-Pope,K. Caldwell-Pope,250,-32,25,Missed,1,NaN,NaN,0,v,MISS Caldwell-Pope 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,4
4,0022100006,10,PT11M22.00S,1,1610612761,TOR,1628384,Anunoby,O. Anunoby,0,0,0,NaN,0,NaN,NaN,0,h,Anunoby REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
642472,0042100311,621,PT00M42.60S,4,1610612744,GSW,202357,Bjelica,N. Bjelica,0,0,0,NaN,0,NaN,NaN,0,h,Bjelica REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,458
642473,0042100311,622,PT00M35.00S,4,1610612744,GSW,1627814,Lee,D. Lee,237,14,24,Made,1,112.0,87.0,199,h,Lee 24' 3PT Jump Shot (3 PTS) (Kuminga 1 AST),Made Shot,Jump Shot,1,3,459
642474,0042100311,624,PT00M22.60S,4,1610612742,DAL,203504,Burke,T. Burke,-230,31,0,Missed,1,NaN,NaN,0,v,MISS Burke 3PT Jump Shot,Missed Shot,Jump Shot,1,3,460
642475,0042100311,625,PT00M19.10S,4,1610612744,GSW,1627814,Lee,D. Lee,0,0,0,NaN,0,NaN,NaN,0,h,Lee REBOUND (Off:0 Def:5),Rebound,Unknown,1,0,461


In [13]:
s26 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S25.csv')
s26.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,22500086,2,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.0,0.0,0,NaN,Start of 1st Period (8:14 PM EST),period,start,1,0,1
1,22500086,4,PT12M00.00S,1,1610612749,MIL,1626167,Turner,M. Turner,0,0,0,NaN,0,NaN,NaN,0,h,Jump Ball Turner vs. Sarr: Tip to McCollum,Jump Ball,NaN,1,0,2
2,22500086,7,PT11M50.00S,1,1610612749,MIL,1631260,Green,A. Green,0,0,0,NaN,0,NaN,NaN,0,h,Green P.FOUL (P1.T1) (B.Forte),Foul,Personal,1,0,3
3,22500086,9,PT11M42.00S,1,1610612764,WAS,203114,Middleton,K. Middleton,130,-5,13,Missed,1,NaN,NaN,0,v,MISS Middleton 13' Turnaround Fadeaway Shot,Missed Shot,Turnaround Fadeaway shot,1,2,4
4,22500086,10,PT11M38.00S,1,1610612764,WAS,1642259,Sarr,A. Sarr,0,0,0,NaN,0,NaN,NaN,0,v,Sarr REBOUND (Off:1 Def:0),Rebound,Unknown,1,0,5


In [ ]:
from src.features.feature_engineer.pbp_features import _get_location

def build_player_quarter_features(pbp_df, player_id, quarter):
    player_q  = pbp_df[(pbp_df["personId"] == player_id) & (pbp_df["period"] == quarter)]
    full_q    = pbp_df[pbp_df["period"] == quarter]
    team_id   = player_q["teamId"].iloc[0]
    team_q    = full_q[full_q["teamId"] == team_id]
    opp_q     = full_q[full_q["teamId"] != team_id]
    location  = _get_location(pbp_df, player_id)
    player_name = player_q["playerName"].iloc[0]

    mins   = get_minutes_played(full_q, player_name, team_id, quarter)
    shots  = get_shooting_stats(player_q, mins["minutes_played_in_q"])
    
    return {
        **mins,
        **shots,
        **get_shot_geography(player_q),
        **get_usage(player_q, team_q, mins["minutes_played_in_q"],
                    shots["FGA"], shots["FTA"], get_other_stats(player_q)["turnovers"]),
        **get_game_context(full_q, location),
        **get_scoring_rhythm(player_q),
        **get_transition_scoring(player_q, opp_q),
        **get_momentum_features(player_q, full_q, location),
        **get_temporal_distribution(player_q, full_q, location),
        **get_other_stats(player_q),
    }

In [ ]:
game_df = s26[s26['gameId'] == 22500086]
game_df = cleanPlaybyPlay(game_df)
build_player_quarter_features(game_df, player_id=203114, quarter=1)